In [1]:
from math import exp
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0, # ตั้งค่าเป็น 0 เพื่อความแม่นยำสูงสุด
    model_kwargs={
        "logprobs": True # เปิดใช้งาน logprobs
    }
)

C:\Users\User\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3687: UserWarning: Parameters {'logprobs'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [3]:
def check_allergy_system(menu_item, allergy_list):
    prompt = f"""ตรวจสอบว่าเมนู "{menu_item}" มีส่วนผสมที่คนแพ้ "{allergy_list}" หรือไม่
กฎการตอบ:
- ถ้ามี/เสี่ยง ให้ตอบว่า "True"
- ถ้าไม่มี/ปลอดภัย ให้ตอบว่า "False"
คำตอบ:"""

    response = llm.invoke([HumanMessage(content=prompt)])

    print("Metadata:", response.response_metadata)
    
    try:
        logprob_data = response.response_metadata.get("logprobs", {}).get("content", [])
        if logprob_data:
            chosen_logprob = logprob_data[0]["logprob"]
            confidence = exp(chosen_logprob) # สูตร P = e^logprob 
        else:
            confidence = -1.0
    except (KeyError, IndexError, TypeError):
        confidence = -1.0

    answer = response.content.strip().lower()

    # Human-in-the-loop
    if 0.1 <= confidence <= 0.9:
        status = "MANUAL_REVIEW"
    elif answer == "true":
        status = "ALLERGY_WARN"
    else:
        status = "SAFE"

    return {
        "answer": answer,
        "confidence": f"{confidence:.2%}",
        "status": status
    }

In [4]:
result = check_allergy_system("คาเปรเซ่สลัด", "นมควาย")
print(f"AI ตอบ: {result['answer']} | ความมั่นใจ: {result['confidence']}")
print(f"สรุปสถานะ: {result['status']}")

Metadata: {'token_usage': {'completion_tokens': 1, 'prompt_tokens': 77, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DKF74Ac09U40sEeuDHwHSf6xzspGt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': {'content': [{'token': 'True', 'bytes': [84, 114, 117, 101], 'logprob': -0.3609026372432709, 'top_logprobs': []}], 'refusal': None}}
AI ตอบ: true | ความมั่นใจ: 69.70%
สรุปสถานะ: MANUAL_REVIEW
